In [6]:
# Localization_Regression.ipynb

import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import vision_models.constants as constants
from pydicom.pixel_data_handlers.util import apply_modality_lut
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from keras import layers, models, applications, losses



In [ ]:

# Set paths
train_images_path = constants.TRAIN_DATA_PATH
train_df = pd.read_csv("NEW_EDA/train_dataset.csv")
val_df = pd.read_csv("NEW_EDA/val_dataset.csv")
test_df = pd.read_csv("NEW_EDA/test_dataset.csv")


In [3]:

# Data Preparation
for df in [train_df, val_df, test_df]:
    df['image_path'] = df.apply(lambda row: os.path.join(train_images_path, str(row['study_id']), str(row['series_id']), f"{row['instance_number']}.dcm"), axis=1)


In [4]:

# Function to load and preprocess images
def load_image(img_path, target_size=(224, 224)):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_dicom_image(img)
    img = tf.image.resize(img, target_size)
    img = img / 255.0  # Normalize pixel values
    return img

# Create a TensorFlow dataset
def create_tf_dataset(df, batch_size=32, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices((df['image_path'], df[['x', 'y']]))
    dataset = dataset.map(lambda x, y: (load_image(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

train_dataset = create_tf_dataset(train_df)
val_dataset = create_tf_dataset(val_df, is_training=False)
test_dataset = create_tf_dataset(test_df, is_training=False)


AttributeError: in user code:

    File "/var/tmp/ipykernel_245339/3359580260.py", line 12, in None  *
        lambda x, y: (load_image(x), y)
    File "/var/tmp/ipykernel_245339/3359580260.py", line 4, in load_image  *
        img = tf.image.decode_dicom_image(img)

    AttributeError: module 'tensorflow._api.v2.image' has no attribute 'decode_dicom_image'


In [ ]:

# Model 1: Basic CNN for Regression
model1 = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='linear')  # Output for x and y coordinates
])

model1.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Model 2: U-Net for Segmentation and Regression
def unet_model(input_size=(224, 224, 1)):
    inputs = layers.Input(input_size)
    c1 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c3)
    
    u4 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u4)
    c4 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c4)
    
    u5 = layers.Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c1], axis=3)
    c5 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(c5)
    
    outputs = layers.Conv2D(2, (1, 1), activation='linear')(c5)
    
    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

model2 = unet_model()
model2.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the models
history1 = model1.fit(train_dataset, validation_data=val_dataset, epochs=10)
history2 = model2.fit(train_dataset, validation_data=val_dataset, epochs=10)

# Evaluate the models
def plot_regression_metrics(history, title):
    plt.plot(history.history['mae'], label='Train MAE')
    plt.plot(history.history['val_mae'], label='Validation MAE')
    plt.title(f'{title} MAE')
    plt.ylabel('MAE')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

plot_regression_metrics(history1, "Model 1 - CNN Regression")
plot_regression_metrics(history2, "Model 2 - U-Net Regression")

# Function to predict and evaluate
def evaluate_regression_model(model, dataset, df, dataset_name):
    predictions = model.predict(dataset)
    true_coords = df[['x', 'y']].values
    mae = mean_absolute_error(true_coords, predictions)
    mse = mean_squared_error(true_coords, predictions)
    print(f"Evaluation on {dataset_name} Data:")
    print(f"MAE: {mae}, MSE: {mse}")

# Evaluate on Validation and Test datasets
evaluate_regression_model(model1, val_dataset, val_df, "Validation")
evaluate_regression_model(model1, test_dataset, test_df, "Test")

evaluate_regression_model(model2, val_dataset, val_df, "Validation")
evaluate_regression_model(model2, test_dataset, test_df, "Test")
